# Classification using Naive Bayes
---

In this notebook we will do our first classification experiments with **Naive Bayes**. 
It is used on categorical, numerical and -as an advanced example- on text data.

Also, the basic concepts of separating training and test data, normalization, the accuracy and a confusion matrix are applied.

### Overview
---

- [Naive Bayes Algorithm](#Naive-Bayes-Algorithm)
  - [Naive Bayes using Categorical Data](#Naive-Bayes-using-Categorical-Data)
  - [Gaussian Naive Bayes](#Gaussian-Naive-Bayes)
- [Text Mining with Naive Bayes](#Text-Mining-with-Naive-Bayes)
  - [Counting Words and BernoulliNB](#Counting-Words-and-BernoulliNB)
  - [Comparing with tf-idf](#Comparing-with-tf-idf)

# Naive Bayes Algorithm

## Naive Bayes using Categorical Data
---

In order to apply a first *Naive Bayes* classifier on categorical data, we use a well-known (but very small) dataset, which is about deciding to play golf (or not), depending on the weather conditions.

As always, the necessary imports first:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder 
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score

from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler

Import and display the data:

In [ ]:
golf_df = pd.read_csv("data/golf.csv")
golf_df

The label (playing golf or not) is in the last column, let's store the data in $X$ and the labels in $y$:

In [ ]:
y =  golf_df["Play Golf"]
y

In [ ]:
X = golf_df.iloc[:,0:-1]
X

Since we know we need to have a training and a test set, it is now time to split our data.
Let's decide to use 80% for training and 20% for testing.

**Note:** the test set is very small here, the predicted accuracy will probably be not very significant. (This is just a toy example!)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
         X, y, test_size=0.2, random_state=3) # if you want, you can remove random_state here and run it a bunch of times

In [ ]:
print("Train:")
print(X_train, '\n', y_train)
print('\nTest:')
print(X_test, '\n', y_test)

Please have a look at the IDs and observe the outcome of the train_test_split-method: It makes sure the sets do not overlap.

Unfortunately, the CategoricalNB model from `scikit-learn` can not directly deal with Strings as categories. It is required to use an integer encoding for each category. The `OrdinalEncoder` comes in handy here. As the name implies, it is typically used to convert ordinal data into ordered integers. In our case the data has no order and the integer values are treated as *categories* by Naive Bayes algorithm (e.g. distances do not play a role as you know from the lecture).


In [ ]:
enc = OrdinalEncoder()
enc.fit(X)  # scan data; we use X to make sure all categories are present in the encoder
X_train_enc = enc.transform(X_train)  # encode the training data
X_test_enc = enc.transform(X_test) # encode using the same "fitted model", meaning the order of the data

print(X_test_enc)

Now it is time to create the **Naive Bayes model**. We can do this using a Laplace smoothing (called alpha here) or not.

Afterwards, we train ("fit") it using the training set (with labels)

In [ ]:
# model = CategoricalNB(alpha=0,force_alpha=True) without Laplace triggers a warning, because some probability is zero..
model = CategoricalNB() # default alpha is 1.0, if you want, you could lower it by adding a parameter "alpha=..."
model.fit(X_train_enc,y_train)

The model is now applied on the test set and results are printed:

In [ ]:
pred = model.predict(X_test_enc)

In [ ]:
print(pred)
print(list(y_test)) # compare to ground truth!

Looks like we were in two out of three tests correct, achieving an accuracy of 66%.

It is also possible to have a look into the single (conditional) probabilities of the model. They are stored as logarithmic values (due to performance reason), so it is required to use the `numpy`exp method to see the real values. Please note that for every dimension there is an array with two classes (No, Yes) and the length of the array depends on the amount of different categories. All probabilities need to sum up to ~1.0

In [ ]:
f=0
for i in model.feature_log_prob_:
    f=f+1
    print("Dimension",f,'\n',np.exp(i))
    print()

It is also possible to check the class prior learned by the model:

In [ ]:
print(model.classes_)
print(np.exp(model.class_log_prior_))


In order to evaluate the results more quantitatively, it is possible to compute the **accuracy** and also plot a **confusion matrix**. It is not really meaningful on this (too) small dataset, but just that you know how it is done:

In [ ]:
print('accuracy:', accuracy_score(y_test, pred))

# Plot non-normalized confusion matrix
titles_options = [
    ("Confusion matrix, without normalization", None),
    #("Normalized confusion matrix", "true"),
]
for title, normalize in titles_options:
    disp = ConfusionMatrixDisplay.from_estimator(
        model,
        X_test_enc,
        y_test,
        cmap=plt.cm.Blues,
        normalize=normalize,
    )
    disp.ax_.set_title(title)

    print(title)
    print(disp.confusion_matrix)

plt.show()

The confusion matrix aligns well with our results above. Finally, a last classification example simulating a *deployment* of our model:

In [ ]:
today = pd.DataFrame([['Sunny', 'Mild', 'Normal', False]]) # feel free to adjust the values to today's weather :)
today.columns=X.columns # set column names to avoid warning
today_enc = enc.transform(today) # label encoder
print("Play golf today:",model.predict(today_enc)[0])

Additionally, you might have noticed, there are much more 'Yes' than 'No' items in the training data -- maybe it would be a good idea to balance the training set. Or, alternatively, we could neglect the class prior $P(C_i)$ for our model. Let's check if results are improving that way...

In [ ]:
model = CategoricalNB(fit_prior=False) # P(C1) == P(C2) == 0.5
model.fit(X_train_enc,y_train)
pred2 = model.predict(X_test_enc)

In [ ]:
print(pred2)
print('Accuracy:', accuracy_score(y_test,pred2))

Well, there was no improvement, although the model is now neglecting the class prior.

## Gaussian Naive Bayes
---


In order to use **numerical data** and Naive Bayes, the probability estimates are done based on a **Normal distribution**, aka. Gaussian Naive Bayes.

To this end, the well-know iris dataset is used again. Please note, that it contains three classes of three different species.

We start with importing the data and normalizing it.

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data , columns=iris.feature_names)
print(list(iris.target_names)) # print classes
y = iris.target # target class
df.head()

In [ ]:
scaler = MinMaxScaler() # Normalization is not neccessary in GaussianNB, because indepenence of the features. 
                        # But it could not harm doing it nevertheless ...
scaler.fit(df)
X = scaler.transform(df)
X[:5,:]

In [ ]:
# Train/Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=41)

# Create model and training
model = GaussianNB()
model.fit(X_train, y_train)

# testing
y_pred = model.predict(X_test)

# Accuracy
print('Accuracy is',accuracy_score(y_pred,y_test))

A confusion matrix helps now to check which classes are confused with each other...

In [ ]:
# Plot confusion matrix
titles_options = [
    ("Confusion matrix, without normalization", None),
    ("Normalized confusion matrix", "true"),
]
for title, normalize in titles_options:
    disp = ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        display_labels=iris.target_names,
        cmap=plt.cm.Blues,
        normalize=normalize,
    )
    disp.ax_.set_title(title)

    print(title)
    #print(disp.confusion_matrix)

plt.show()

Typically, a confusion matrix is displayed as on the first figure. The second one shows a "normalized" version, meaning every ground truth row adds up to 1.0 (=100%), that way it is easier to see the misclassifications in percent.

Print the data of the two misclassified instances:

In [ ]:
fail = X_test[y_test!=y_pred] 
print(fail)
print(scaler.inverse_transform(fail)) # undo the MinMaxScaler to see the "real" data

# Text Mining with Naive Bayes
---


## Counting Words and BernoulliNB
Now we will get to know a simple approach to language processing (aka *Text Mining*) in Python, again using `scikit-learn`. In this context, a corresponding preprocessing of the data and *feature extraction* is required. You also have to deal with very *sparse matrices*.

The goal of this example is to recognize insulting posts in social media (e.g. in a forum). The idea is based on a chapter of the [IPython Cookbook](https://ipython-books.github.io/). The data set is available in the `data` folder and was downloaded from the Cookbook's [data repository](https://github.com/ipython-books/cookbook-data). The dataset was originally provided by the company Impermium (now Google) as part of a [Kaggle competition](https://www.kaggle.com/c/detecting-insults-in-social-commentary).

**Note:** For text mining, there are many dedicated packages availabe in Python, which we will not use here for simplicity. The most well-know is [nltk](http://www.nltk.org/).

Some more imports here:

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import sklearn.model_selection as ms
import sklearn.feature_extraction.text as text
import sklearn.naive_bayes as nb

Firstly, the data is imported as a DataFrame from the CSV:

In [ ]:
df = pd.read_csv("data/troll.csv")

Every instance contains a label in the column *Insult* if a comment is insulting (1) or not (0). See the example below:

In [ ]:
df.tail()

Next, the data is separated into the data  $X$ and the target (labels) $y$. Obtaining $y$ is easy:

In [ ]:
y = df['Insult']
y.head()

The feature matrix $X$, on the other hand, is much more difficult to obtain. `scikit-learn` can only deal with tabular data, so we have to convert the text into a matrix. This **data preprocessing** (e.g. the *data preparation* according to [CRISP-DM](https://en.wikipedia.org/wiki/Cross-industry_standard_process_for_data_mining)) usually happens in two steps:

1. Extract a **vocabulary**, a list of all words that were used in the text (in our case: in the comments).
2. You then "count" how often the respective word occurs in each comment. Since there are generally a lot of words and only very few of them are actually used in a specific row (in our case: in a comment), this leads to a matrix that mainly contains zeros (i.e. is very sparse).

In `scikit-learn`. the `CountVectorizer` is used for creating the vocabulary and for "counting":


In [ ]:
co = text.CountVectorizer()  # count vectorizer
Xc_vec = co.fit(df['Comment']) # create vocabulary and count words
Xc = Xc_vec.transform(df['Comment']) # Preprocessing: Turn the text in a (very) high dimensional vector with the length
                                     # of the vocabulary!
# or in one step: Xc = co.fit_transform(df['Comment'])
print(Xc.shape)

There are 3947 rows, with a vector of 16469 dimensions (obvously, we have so many different words). We can check how often a specific word occurs in the comments (sorry about bad words being displayed here):

In [ ]:
list(Xc_vec.vocabulary_.items())[:10]

Please note that the output is not ordered, just a snippet of the words. Probably you can observe that the CountVectorizer automatically eliminated special charcters (don't -> don) and that the encoding problem "\xa0" turned into a new word xa0. Obviously, this pre-processing process could be made much better!

Lets check the sparity/density of the matrix. Since it is very sparse, there is even a specific datatype used (not a regular np array, which would require much more memory:

In [ ]:
p = 100 * Xc.nnz / float(Xc.shape[0] * Xc.shape[1]) # density = non-zero entries / (rows * dimensions)
print(f"X has ~{p:.2f}% non-zero values.")
print(type(Xc))

Let's print the vector with counts for our second document (it is a bit tricky...)

In [ ]:
print(df['Comment'][1])
print(Xc[1])

It seems like the sparce matrix object shows us a "map" of words (IDs) and the count. To see the full 16469 dimensional vector, we can try:

In [ ]:
print(Xc[1].toarray()) # there are dots... not very helpful...

with np.printoptions(threshold=np.inf): # finally!
    print(Xc[1].toarray())

Now the datset is splitted into 80% train set and 20% test set:

In [ ]:
(Xc_train, Xc_test, yc_train, yc_test) = ms.train_test_split(Xc, y, test_size=.2, random_state = 4)
print('Training set:', Xc_train.shape[0])
print('Test set:', Xc_test.shape[0])

Now it is time to apply your Naive Bayes model. To make things a bit faster, we are using the **Bernoulli** Naive Bayes algorithm. It is basicaly the same as the categorical one, but the features are reduced to a binary vector. In short, the counts of the words are reduced to "1" (if the word is present, otherwise 0). Please compare it to the "bag-of-words" vs. "one-hot" in the lecture slides!

In [ ]:
model = nb.BernoulliNB(alpha=0.63) # set alpha to good value; default=1.0 is not working great. (if you want to try..)
model.fit(Xc_train, yc_train) 

# Optional: How to find a good alpha -> hyperparameter tuning; next lecture!
# bmodel = ms.GridSearchCV(nb.BernoulliNB(), param_grid={'alpha':np.logspace(-10, 0., 50)})
# bmodel.fit(Xc_train, yc_train);
# print(bmodel.best_estimator_)

Now we can evaluate using the test data. Either as we did it before or using a shortcut, the `score()`-method of the model.

In [ ]:
print(f'Accuracy. {round(100*model.score(Xc_test, yc_test),2)}%')
# pred = model.predict(Xc_test)
# print('Accuracy:',accuracy_score(yc_test, p))

Not bad, in 79% of the test data we were able to predict whether a comment is insulting or not! Additionally, we can try with some "comments" from our side to test the model...

In [ ]:
print(model.predict(co.transform([
    "I totally agree with you",
    "You are so stupid",
    "lol"
])))

Finally, we could again plot the confusion matrix:

In [ ]:
# Plot confusion matrix
titles_options = [
    ("Confusion matrix, without normalization", None),
    #("Normalized confusion matrix", "true"),
]
for title, normalize in titles_options:
    disp = ConfusionMatrixDisplay.from_estimator(
        model,
        Xc_test,
        yc_test,
        display_labels=[0,1],
        cmap=plt.cm.Blues,
        normalize=normalize,
    )
    disp.ax_.set_title(title)

    print(title)
    #print(disp.confusion_matrix)

plt.show()

**Note**: Please be aware, that the Naive Bayes classificator takes the class priors into account. We can see from the confusion matrix, that there are much more regular than insulting comments (we should have checked in the data understanding earlier...). So the result, having more insulting comments being classified as regular might be due to that. Can you think of a way to check whether this is true?

## Comparing with tf-idf

With the help of `scikit-learn` we only need two lines of code for this. The [tf-idf](https://de.wikipedia.org/wiki/Tf-idf-Ma%C3%9F) method is used as discussed in the lecture for "counting", with the help of which words that occur too frequently ("the" , "and, etc.) can be adequately treated. Let's use the `TfidfVectorizer`:

In [ ]:
tf = text.TfidfVectorizer()  # tf-idf 
X_vec = tf.fit(df['Comment'])
X = X_vec.transform(df['Comment'])
print(X.shape)

We get exactly the same size as before (no wonder because of the same vocabulary). Also, the sparcity is the same (in case you want to verify that), but the vector looks now a bit different:

In [ ]:
X.max() # max value is now no count but ..

In [ ]:
with np.printoptions(threshold=np.inf):
    print(X[1].toarray())

Split the dataset as before (and also with the same `random_state`to have exactly the same train/test-sets as earlier:

In [ ]:
(X_train, X_test, y_train, y_test) = ms.train_test_split(X, y, test_size=.2, random_state = 4)

And train and evaluate the Bernoulli Naive Bayes again:

In [ ]:
modelBer = nb.BernoulliNB(alpha=0.63) # set alpha to good value, same as above.
modelBer.fit(X_train, y_train)

print(f'Accuracy. {round(100*modelBer.score(Xc_test, yc_test),2)}%')

Actually we get **exactly** the same result as before -- but no wonder! Do you remember how the Bernoulli Naive Bayes works? It binarizes the vector to zero and ones, so we have exactly the same data.

Maybe a classifier, which uses numerical values, may be useful. We only know GaussianNB so far, so let's try it:

In [ ]:
model = nb.GaussianNB()
model.fit(X_train.toarray(), y_train) # sparse not working here, so we need a bit of memory...
print(f'Accuracy. {round(100*model.score(X_test.toarray(), y_test),2)}%')

How disappointing! Obviously this works much worse... Actually this is not surprising, because the data within a single dimension is far from being a normal distribution. So far, it seems like tf-idf is not helping much with Naive Bayes compared to simple counting.

However, do you remeber the *Ridge Regression* from lecture 2? Actually it was based on a linear model (least squares), and it can also be used as a classifier, in `scikit-learn` it is called `RidgeClassifier`. In comparison to more complex models, which we will deal with in the future lectures, the `RidgeClassifier` is also fast and applicable on this data due to its linear nature. Let's just try it out:

In [ ]:
from sklearn.linear_model import RidgeClassifier

model = RidgeClassifier().fit(X_train.toarray(), y_train)
print(f'Accuracy. {round(100*model.score(X_test.toarray(), y_test),2)}%')

It performs better! At least a bit... Did you notice the longer computation time compared to BernoulliNB?

Finally, let's perform a last experiment: Can we find out, **which words** yield to a decision telling us a comment is "regular" or "insulting"? Actually, we can, not with the `RidgeClassifier`, but with NaiveBayes! For now, I am using again the Bernoulli Naive Bayes, where the model was stored in the variable *modelBer* above. The idea is simple: We get the probabilities $P(x_i|C_0)$ and $P(x_i|C_1)$ and sort it by the value. Words with a high probability are relevant or a strong indicator for a specific class:

In [ ]:
# get the words corresponding to the IDs
names = np.asarray(tf.get_feature_names_out()) # in case you have an older version of scikit-lean: use get_feature_names()

print('Good words (top40):')
print(','.join(names[np.argsort(
    modelBer.feature_log_prob_[0,:])[::-1][:40]])) # remeber feature_log_prob_ from the CategoricalNB?
print()
print('Bad words (top40):')
print(','.join(names[np.argsort(
    modelBer.feature_log_prob_[1,:])[::-1][:40]]))

Interestingly, common words like 'the','you', ... still show up with a high probability, although tf-idf has probably "punished" their weight in the feature vector already...

However, in the list of bad words, there are some obvious cases.